In [1]:
%matplotlib inline
import random 
import torch
from d2l import torch as d2l

In [2]:
def synthetic_data(w,b,number_example):
    "y = Xw+b+噪声"
    X = torch.normal(0,1,(number_example,len(w)))
    y = torch.matmul(X,w)+b   #matrix multiplication  这里外面不能加括号先矩阵乘法再加b 
    y += torch.normal(0,0.01,y.shape)
    return X,y.reshape(-1,1)

In [3]:
true_w =torch.tensor([2,-3.4]) 
true_b =4.2
features ,labels=synthetic_data(true_w,true_b,1000)
print("样本特征",features[:10],"\n","答案",labels[:10])

样本特征 tensor([[ 0.4462,  1.4548],
        [-0.7654, -0.1630],
        [ 1.0428,  0.4477],
        [-0.3268,  0.1449],
        [ 0.5876,  0.2287],
        [-1.3286, -0.9738],
        [ 0.3589,  0.1311],
        [-1.3760,  0.0417],
        [-0.2498,  0.5270],
        [-0.4351,  0.0641]]) 
 答案 tensor([[0.1474],
        [3.2046],
        [4.7422],
        [3.0489],
        [4.6070],
        [4.8502],
        [4.4586],
        [1.3020],
        [1.8955],
        [3.1234]])


In [4]:
def data_iter (batch_size,features,labels):
    " "
    num_examples = len(features)
    print("样本个数",num_examples)
    indices = list(range(num_examples))
    print("生成索引个数",indices[:10])
    random.shuffle(indices)
    print("洗牌索引个数",indices[:10])
    for i in range (0,num_examples,batch_size):
        batch_indices = torch.tensor(indices[i:min(i+batch_size,num_examples)])   #indices[0:9] 从indices这个列表里取一段数 indices[10:19]  min(10,3) min是在两个值里取哪个值更小 
        yield features[batch_indices],labels[batch_indices];

batch_size = 10 
for x,y in data_iter(batch_size,features,labels):
    print("每一批次的x:",x,'\n',"每一批次的y:",y)
    break

样本个数 1000
生成索引个数 [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
洗牌索引个数 [635, 899, 783, 623, 241, 295, 894, 846, 534, 748]
每一批次的x: tensor([[-1.2508, -0.5531],
        [ 1.5164, -0.3182],
        [-0.5972, -0.4379],
        [ 0.6728, -1.0918],
        [ 2.0039,  2.6341],
        [ 0.0823,  1.9769],
        [-0.1939, -0.3514],
        [-0.6891, -0.7513],
        [ 0.2825,  1.1048],
        [ 1.0243,  1.1593]]) 
 每一批次的y: tensor([[ 3.5786],
        [ 8.3107],
        [ 4.4882],
        [ 9.2523],
        [-0.7546],
        [-2.3659],
        [ 5.0218],
        [ 5.3701],
        [ 1.0114],
        [ 2.3015]])


In [5]:
#初始化，给w和b一个从哪里开始猜 
w = torch.normal(0,0.01,size=(2,1),requires_grad=True)
b = torch.zeros(1,requires_grad = True)
print("w:",w,"\n","b:",b)

w: tensor([[3.6427e-03],
        [4.0858e-05]], requires_grad=True) 
 b: tensor([0.], requires_grad=True)


In [6]:
def linreg(X,w,b):
    """线性回归"""
    y_hat = torch.matmul(X,w)+b
    return y_hat 

In [7]:
def squared_loss(y_hat,y):
    """均方损失"""
    loss=(y_hat-y.reshape(y_hat.shape))**2/2
    return loss

In [8]:
def sgd(params,lr,batch_size):
    "stochastic gradinet descent"
    "小批量随机梯度下降"
    "∂L/∂W ，∂L/∂/b"
    with torch.no_grad():
        for param in params:
            param -= ((lr*param.grad)/batch_size)
            param.grad.zero_()
    
    

In [14]:
lr = 0.03 
num_epochs = 3


for epoch in range (num_epochs):
    for x,y in data_iter (batch_size,features,labels):
        l = squared_loss(linreg(x,w,b),y)
        l.sum().backward()  #这里已经把10个数的梯度都求出来了
        print("这批次的梯度:",w.grad.flatten().tolist(),"b:",b.grad.item()) 
        print("before:", w, b)
        sgd([w,b] ,lr,batch_size)  #这里的w,b为什么可以凭空而来 我用随机梯度下降更新了我的w ,
        print("after:", w, b)
        with torch.no_grad():
            train_l = squared_loss(linreg(features,w,b),labels) #这里是算我们当前的损失分数
    #print("train_l:",train_l)
    print (f'epoch{epoch+1},loss{float(train_l.mean()):f}')
            
        
        
    


样本个数 1000
生成索引个数 [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
洗牌索引个数 [373, 885, 983, 83, 419, 9, 266, 304, 105, 886]
这批次的梯度: [0.030935723334550858, 0.043429240584373474] b: 0.0021101534366607666
before: tensor([[ 1.9993],
        [-3.3996]], requires_grad=True) tensor([4.2005], requires_grad=True)
after: tensor([[ 1.9992],
        [-3.3998]], requires_grad=True) tensor([4.2005], requires_grad=True)
这批次的梯度: [0.10158678144216537, -0.052298299968242645] b: -0.01084446907043457
before: tensor([[ 1.9992],
        [-3.3998]], requires_grad=True) tensor([4.2005], requires_grad=True)
after: tensor([[ 1.9989],
        [-3.3996]], requires_grad=True) tensor([4.2006], requires_grad=True)
这批次的梯度: [0.014521006494760513, -0.026397572830319405] b: 0.06698644161224365
before: tensor([[ 1.9989],
        [-3.3996]], requires_grad=True) tensor([4.2006], requires_grad=True)
after: tensor([[ 1.9989],
        [-3.3995]], requires_grad=True) tensor([4.2004], requires_grad=True)
这批次的梯度: [-0.004361974075436592, -0.004187169